# Урок 3. Столбцы, вычисления и изменения

**Курс:** pandas от нуля до профессионала · **Модуль 01** · Урок 3 из 4

**Как работать:** внимательно читайте теорию, запускайте демонстрационные ячейки, затем выполняйте задания без копирования примеров. У каждого задания есть отдельная ячейка проверки на показанных исходных данных.

> Для заданий вам понадобятся только знания из этого и предшествующих уроков. Интернет не требуется.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
from tests.notebook_checks import expected_for, check_result, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Выбор столбцов

Выражение `df["column"]` возвращает Series. Выражение `df[["a", "b"]]` возвращает DataFrame с двумя выбранными столбцами **в указанном порядке**.

```python
import pandas as pd
orders = pd.DataFrame({"id": [1,2], "qty": [3,4], "price": [80,120]})
print(orders[["id", "price"]])
```

При выборе отсутствующей колонки возникает `KeyError`. Перед сложной обработкой полезно проверить требуемые столбцы: `set(required).issubset(df.columns)`. Индексацию по строкам (`loc`, `iloc`) разберём во втором модуле.

## 2. Векторные вычисления

В pandas арифметика выполняется **поэлементно**: каждый элемент одного столбца взаимодействует с соответствующим элементом другого.

```python
invoice = pd.DataFrame({"units": [2, 5], "unit_price": [70, 12]})
invoice["cost"] = invoice["units"] * invoice["unit_price"]
invoice
```

Скаляр также применяется к каждому элементу: `df["price"] * 1.1`. Обычно не нужен цикл `for` по строкам — векторный код компактнее и часто быстрее.

**Осторожно:** операции по двум Series выравниваются по индексам. При делении возникают `inf`/`NaN`, если знаменатель ноль или данные отсутствуют; проверку таких ситуаций разберём позже.

## 3. Создание и удаление столбцов

`df["new"] = ...` добавляет или заменяет столбец **в исходной таблице**. Если хотите сохранить исходные данные, начните с `result = df.copy()` или используйте `df.assign(...)`, который возвращает новую таблицу.

```python
t = pd.DataFrame({"x": [2, 4], "y": [3, 5]})
with_total = t.assign(total=t["x"] + t["y"])
print(t.columns.tolist())            # только x, y
print(with_total.columns.tolist())   # x, y, total
```

Для удаления столбца: `df.drop(columns=["col"])`. Без переназначения переменной исходная таблица обычно остаётся прежней. Не злоупотребляйте `inplace=True`: явно возвращаемые данные легче тестировать и читать.

## 4. Переименование и сортировка

`rename(columns={"old": "new"})` создаёт таблицу с изменёнными именами. `sort_values(by="amount")` сортирует по значениям; `ascending=False` меняет направление. Сортировка по нескольким столбцам задаётся списком: `by=["category", "amount"]`, а направление можно указать как `ascending=[True, False]`.

```python
r = pd.DataFrame({"group": ["b", "a", "a"], "score": [3, 2, 9]})
print(r.sort_values(by=["group", "score"], ascending=[True, False]))
```

После сортировки старые метки индекса обычно сохраняются. Это нормально: сортировка меняет порядок строк, не обязательно их метки. Для нумерации от нуля можно вызвать `.reset_index(drop=True)`, когда это действительно нужно.

## 5. Частые ошибки

- Пытаться присвоить значения столбцу с несовпадающим индексом Series: они выровняются по меткам, а не по порядку.
- Изменять входной DataFrame внутри функции, если задача требует сохранить его неизменным.
- Случайно получать Series вместо DataFrame из-за одной пары квадратных скобок.
- Путать удаление столбцов с удалением строк: `drop(columns=...)` явно работает с колонками.
- Считать, что `sort_values()` всегда меняет таблицу в исходной переменной.

Перед заданиями объясните себе: чем `df.assign(total=...)` принципиально отличается от `df["total"] = ...`?

## Дополнительный разбор: вычисляемые столбцы и побочные эффекты
Преобразование вида `df.assign(total=df.qty * df.price)` возвращает новый DataFrame. Это упрощает повторную проверку: исходная таблица остаётся пригодной для других расчётов. В функции лучше явно возвращать таблицу, а не рассчитывать на то, что вызывающая сторона заметит незаметные изменения.

В pandas операции над Series векторизованы — одно выражение создаёт значения для всех строк. Не нужен цикл `for` для обычного умножения и скидок. Следите за единицами измерения: скидка 15% означает множитель `(1 - 15/100)`, а не `1 - 15`. Сортировка может менять порядок строк, но сохраняет индекс, поэтому для отчёта с привычной нумерацией отдельно требуется `reset_index(drop=True)`.

Умение точно сформулировать контракт результата — часть профессии: какие колонки, их порядок, нужно ли сохранять индекс, допускается ли менять оригинал, как обрабатываются пропуски.

---
## Практические задания

**Правила:** выполните ячейку исходных данных, затем пишите обычный код в ячейке решения. Сохраните полученный результат в переменной `result`. Сразу под ней находится **ожидаемый вывод**. Сравните его со своим результатом, затем запустите автопроверку. Функции объявлять не нужно.

**Почему задания отличаются от теории:** в теории показаны простые иллюстрации методов, а здесь другая предметная область и дополнительные условия. Автопроверка сравнивает ваше решение с результатом на приведённых данных.

### Задание 1 · m1_11

Получите новый DataFrame только с колонками из списка `fields`, сохранив заданный порядок. Не меняйте исходную таблицу.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `code`, `price`, `brand`.
- `fields` — список названий столбцов; тип: `list`.


**Выполнение:** используйте готовые переменные `df`, `fields`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m1_11: используются переменные df, fields

df = pd.DataFrame({
    'code': ['A', 'B'],
    'price': [32, 9],
    'brand': ['X', 'Y'],
})

fields = ['brand', 'code']

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m1_11 = expected_for("m1_11", df=df, fields=fields)


In [ ]:
# m1_11: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, fields
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 2 столбца; колонки: `brand`, `code`.

```text
  brand code
0     X    A
1     Y    B
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m1_11", result, _expected_m1_11)

### Задание 2 · m1_12

Получите копию `df` с новым столбцом `revenue`, равным `units * price`. Исходный `df` не изменяйте.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `units`, `price`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m1_12: используются переменные df

df = pd.DataFrame({
    'units': [3, 6],
    'price': [17.0, 12.0],
})

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m1_12 = expected_for("m1_12", df=df)


In [ ]:
# m1_12: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 3 столбца; колонки: `units`, `price`, `revenue`.

```text
   units  price  revenue
0      3   17.0     51.0
1      6   12.0     72.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m1_12", result, _expected_m1_12)

### Задание 3 · m1_13

Получите копию таблицы с новым столбцом `final_price`. Рассчитайте его из `price` и процентной скидки `discount_pct` (например, 20 означает 20%).

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `price`, `discount_pct`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m1_13: используются переменные df

df = pd.DataFrame({
    'price': [100.0, 45.0],
    'discount_pct': [15.0, 20.0],
})

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m1_13 = expected_for("m1_13", df=df)


In [ ]:
# m1_13: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 3 столбца; колонки: `price`, `discount_pct`, `final_price`.

```text
   price  discount_pct  final_price
0  100.0          15.0         85.0
1   45.0          20.0         36.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m1_13", result, _expected_m1_13)

### Задание 4 · m1_14

Получите новую таблицу: переименуйте `old_name` в `new_name`, затем удалите колонку `obsolete`. Остальные колонки сохраните.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `old_code`, `temp`, `name`.
- `old_name` — входной параметр (значение видно в примере); в этом примере: `'old_code'`.
- `new_name` — входной параметр (значение видно в примере); в этом примере: `'sku'`.
- `obsolete` — входной параметр (значение видно в примере); в этом примере: `'temp'`.


**Выполнение:** используйте готовые переменные `df`, `old_name`, `new_name`, `obsolete`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m1_14: используются переменные df, old_name, new_name, obsolete

df = pd.DataFrame({
    'old_code': [1, 2],
    'temp': [2, 3],
    'name': ['A', 'B'],
})

old_name = 'old_code'

new_name = 'sku'

obsolete = 'temp'

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m1_14 = expected_for("m1_14", df=df, old_name=old_name, new_name=new_name, obsolete=obsolete)


In [ ]:
# m1_14: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, old_name, new_name, obsolete
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 2 столбца; колонки: `sku`, `name`.

```text
   sku name
0    1    A
1    2    B
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m1_14", result, _expected_m1_14)

### Задание 5 · m1_15

Отсортируйте `df` по `department` по возрастанию и по `quantity` по убыванию внутри отдела. Получите таблицу с исходными метками индекса.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `department`, `quantity`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m1_15: используются переменные df

df = pd.DataFrame({
    'department': ['B', 'A', 'A'],
    'quantity': [5, 1, 9],
}, index=[11, 13, 12])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m1_15 = expected_for("m1_15", df=df)


In [ ]:
# m1_15: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 2 столбца; колонки: `department`, `quantity`.

```text
   department  quantity
12          A         9
13          A         1
11          B         5
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m1_15", result, _expected_m1_15)

---
## Рефлексия

1. Какие методы и свойства вы сегодня применяли?
2. Какие ошибки возникали и почему?
3. Что изменится, если входная таблица содержит больше строк или иной порядок значений?

Переходите к следующему уроку, когда все задания текущего прошли проверку.